# Signed manifests and explicit signer trust
Generate temporary demonstration keys, sign a synthetic bundle and verify it with a separately pinned trust policy. Keys are deleted on cleanup. Production keys need an approved secret workflow; never reuse this demonstration passphrase.

In [ ]:
from pathlib import Path
import tempfile
import json
root = Path.cwd()
if not (root / 'examples').exists():
    root = root.parent
assert (root / 'examples/parser_samples.json').exists(), 'Run from the repository or notebooks directory'
from timeline_demo.pipeline import Input, run_pipeline, read_timeline
from timeline_demo.core.manifest import verify_bundle
work = tempfile.TemporaryDirectory()
workdir = Path(work.name)
bundle = workdir / 'bundle'
manifest = run_pipeline([
    Input('cloudtrail', root / 'examples/raw/aws/cloudtrail_real_sample.json'),
    Input('entra_signin', root / 'examples/raw/entra/entra_signin_real_sample.jsonl'),
    Input('crowdstrike_detection', root / 'examples/raw/edr/crowdstrike_detection_real_sample.json'),
], bundle, 'notebook-demo')


In [ ]:
from timeline_demo.signing import generate_keypair, public_entry, write_trust, sign_artifact, verify_signature
from timeline_demo.parsers.common import file_hash
password = b'synthetic-notebook-password-only'
keys = generate_keypair(workdir/'keys', password)
key_id, entry = public_entry(keys['public_key'])
policy = {'version':'1.0', 'keys':{key_id:entry}}
trust = workdir/'trust-v1.json'
trust_pin = write_trust(trust, policy)['trust_store_sha256']
signature = workdir/'bundle.sig.json'
source_pin = file_hash(bundle/'audit_manifest.json')
sign_artifact(bundle, keys['private_key'], password, signature, trust, trust_pin)
verify_signature(bundle, signature, trust, trust_pin)

The trust policy and its hash must be obtained through an independent, controlled channel. The signature contains a key fingerprint, not a self-authorizing public key. No trusted signing time, source authenticity or completeness is asserted.

In [ ]:
policy['keys'][key_id]['status'] = 'verify_only'
retired = workdir/'trust-v2.json'
retired_pin = write_trust(retired, policy)['trust_store_sha256']
assert verify_signature(bundle, signature, retired, retired_pin)['key_status'] == 'verify_only'
policy['keys'][key_id]['status'] = 'revoked'
revoked = workdir/'trust-v3.json'
revoked_pin = write_trust(revoked, policy)['trust_store_sha256']
try:
    verify_signature(bundle, signature, revoked, revoked_pin)
except ValueError as error:
    print(type(error).__name__, str(error))
else:
    raise AssertionError('Revoked signer was accepted')
assert file_hash(bundle/'audit_manifest.json') == source_pin

A verify_only key can verify historical attestations, but this signing command refuses to create new ones. Without a trusted timestamp, verification cannot establish when a signature was made. A compromised key must be revoked; revoked signatures are rejected regardless of claimed age. Historical Delta verification receipts remain audit records, not current authorization. See docs/SIGNING.md for rotation and Databricks/Tines deployment.

In [ ]:
work.cleanup()